# Component I: Product Review Generator (E-Commerce)

# (Install & Import)

In [1]:
!pip install transformers datasets accelerate -q

import torch, math
from transformers import (
    GPT2LMHeadModel, GPT2Tokenizer, Trainer,
    TrainingArguments, DataCollatorForLanguageModeling, set_seed
)
from datasets import Dataset

set_seed(42)

## Load Pre-trained GPT-2 Model

In [2]:
tokenizer = GPT2Tokenizer.from_pretrained('distilgpt2')
model = GPT2LMHeadModel.from_pretrained('distilgpt2')

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## Generate Baseline Reviews (Before Fine-Tuning)

In [3]:
def generate_text(model, tokenizer, prompt, max_length=100):
    model.eval()
    inputs = tokenizer.encode(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        output = model.generate(
            inputs,
            max_length=max_length,
            temperature=0.8,
            top_k=50,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)


review_prompts = [
    "This product is",
    "I bought this phone and",
    "The quality of this item"
]

print("=== BASELINE REVIEWS ===\n")
baseline = {}

for p in review_prompts:
    baseline[p] = generate_text(model, tokenizer, p)
    print(f"Prompt: {p}\nOutput: {baseline[p]}\n")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


=== BASELINE REVIEWS ===

Prompt: This product is
Output: This product is not designed for a specific purpose.



























































































Prompt: I bought this phone and
Output: I bought this phone and I can't wait to get it done!"

"I just want to apologize, but I still don't know if I'll stop paying for the phone."
"Thank you for it! Do you have any of the phone you get with me? Can't I be honest and tell you my phone got lost in the woods?"
I finally went to the phone, and my phone was ringing.
"Oh wait! I got it! I got the phone

Prompt: The quality of this item
Output: The quality of this item will depend on your current level of use.

























































































## Prepare Dataset for Fine-Tuning

In [4]:
corpus = [
    "this phone has an amazing battery life and the camera quality is outstanding for the price.",
    "i bought this laptop for college and it handles all my assignments and coding projects perfectly.",
    "the sound quality of these headphones is incredible with deep bass and clear vocals.",
    "this smartwatch tracks my steps accurately and the heart rate monitor is very reliable.",
    "great wireless earbuds with noise cancellation that blocks out all background sound.",
    "the keyboard feels very comfortable for long typing sessions and the backlight is a nice touch.",
    "this portable charger saved me during travel and it charges my phone three times on a single charge.",
    "the tablet screen is bright and colorful which makes watching movies a great experience.",
    "i love this fitness tracker because it motivates me to reach my daily exercise goals.",
    "this bluetooth speaker is compact but delivers surprisingly loud and clear audio.",
    "the delivery was fast and the product was packed securely with no damage at all.",
    "excellent value for money and the build quality feels premium despite the affordable price.",
    "the customer service team was very helpful when i had questions about the product features.",
    "this camera takes stunning photos in low light and the video recording quality is very smooth.",
    "i have been using this product for three months and it still works perfectly like day one.",
    "the design is sleek and modern and it looks great on my desk next to my other gadgets.",
    "easy to set up right out of the box and the instructions were clear and simple to follow.",
    "highly recommend this product to anyone looking for quality and reliability at a fair price.",
    "the software updates keep adding new features which makes this purchase even more worthwhile.",
    "best purchase i made this year and i would definitely buy from this brand again."
]

dataset = Dataset.from_dict({"text": corpus})

tokenized = dataset.map(
    lambda x: tokenizer(x["text"], truncation=True, max_length=128, padding="max_length"),
    batched=True,
    remove_columns=["text"]
)

split = tokenized.train_test_split(test_size=0.15, seed=42)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

## Fine-Tune GPT-2 Model

In [5]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./gpt2-reviews",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=10,
    save_strategy="no",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    data_collator=data_collator
)

trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,4.055469
20,2.899389
30,2.363605
40,1.903479
50,1.675250


TrainOutput(global_step=50, training_loss=2.579438362121582, metrics={'train_runtime': 4.4014, 'train_samples_per_second': 38.624, 'train_steps_per_second': 11.36, 'total_flos': 5552555950080.0, 'train_loss': 2.579438362121582, 'epoch': 10.0})

## Generate Reviews After Fine-Tuning

In [6]:
eval_res = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_res['eval_loss']):.2f}\n")

print("=== FINE-TUNED REVIEWS ===\n")

for p in review_prompts:
    ft = generate_text(model, tokenizer, p)
    print(f"Prompt: {p}")
    print(f"Baseline:   {baseline[p][:120]}")
    print(f"Fine-Tuned: {ft[:120]}\n")

Perplexity: 26.59

=== FINE-TUNED REVIEWS ===

Prompt: This product is
Baseline:   This product is not designed for a specific purpose.




































































Fine-Tuned: This product is powered by Skimlinks MTBF with a video tag that helps us keep track of the fitness and fitness of our us

Prompt: I bought this phone and
Baseline:   I bought this phone and I can't wait to get it done!"

"I just want to apologize, but I still don't know if I'll stop pa
Fine-Tuned: I bought this phone and I'm looking forward to it again and will buy it again. The screen is bright and the screen is br

Prompt: The quality of this item
Baseline:   The quality of this item will depend on your current level of use.






















































Fine-Tuned: The quality of this item has not changed significantly over the past year or so. The purchase was made in the US and fro



# Component II: Recipe Generator (Food-Tech)

# Fine-Tuning GPT-2 for Recipe Generation

In [7]:
tokenizer2 = GPT2Tokenizer.from_pretrained('distilgpt2')
model2 = GPT2LMHeadModel.from_pretrained('distilgpt2')

tokenizer2.pad_token = tokenizer2.eos_token
model2.config.pad_token_id = tokenizer2.eos_token_id

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Generate Baseline Recipes

In [8]:
recipe_prompts = [
    "To make butter chicken",
    "For pasta carbonara",
    "To prepare a chocolate cake"
]

baseline2 = {}

print("=== BASELINE RECIPES ===\n")

for p in recipe_prompts:
    baseline2[p] = generate_text(model2, tokenizer2, p)
    print(f"Prompt: {p}\nOutput: {baseline2[p]}\n")

=== BASELINE RECIPES ===

Prompt: To make butter chicken
Output: To make butter chicken.

This post may contain links to Amazon or other partners; your purchases via these links can benefit Serious Eats. Read more about our affiliate linking policy.

Prompt: For pasta carbonara
Output: For pasta carbonara pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta

Prompt: To prepare a chocolate cake
Output: To prepare a chocolate cake. For a cookie, go to the baking sheet, place

## Prepare Recipe Dataset

In [9]:
recipes = [
    "to make butter chicken start by marinating chicken pieces in yogurt with spices.",
    "heat butter and fry onions until golden then add ginger garlic paste.",
    "add tomato puree and cook until oil separates.",
    "add chicken and cook until done.",
    "finish with cream and serve hot.",

    "for pasta carbonara boil pasta until al dente.",
    "fry pancetta until crispy.",
    "mix eggs cheese and pepper.",
    "combine pasta with pancetta and egg mixture.",
    "serve immediately with parmesan.",

    "to prepare stir fry heat oil in wok.",
    "add vegetables and stir for few minutes.",
    "add sauces and cook well.",
    "serve with rice."
]

dataset2 = Dataset.from_dict({"text": recipes})

tok2 = dataset2.map(
    lambda x: tokenizer2(x["text"], truncation=True, max_length=128, padding="max_length"),
    batched=True,
    remove_columns=["text"]
)

split2 = tok2.train_test_split(test_size=0.15, seed=42)

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

## Fine-Tune Recipe Model

In [10]:
collator2 = DataCollatorForLanguageModeling(tokenizer=tokenizer2, mlm=False)

args2 = TrainingArguments(
    output_dir="./gpt2-recipes",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=10,
    save_strategy="no",
    fp16=torch.cuda.is_available()
)

trainer2 = Trainer(
    model=model2,
    args=args2,
    train_dataset=split2["train"],
    eval_dataset=split2["test"],
    data_collator=collator2
)

trainer2.train()

Step,Training Loss
10,4.805531
20,3.291868
30,2.751090


TrainOutput(global_step=30, training_loss=3.616163126627604, metrics={'train_runtime': 1.7854, 'train_samples_per_second': 61.61, 'train_steps_per_second': 16.803, 'total_flos': 3592830320640.0, 'train_loss': 3.616163126627604, 'epoch': 10.0})

## Generate Recipes After Fine-Tuning

In [11]:
eval2 = trainer2.evaluate()
print(f"Perplexity: {math.exp(eval2['eval_loss']):.2f}\n")

print("=== FINE-TUNED RECIPES ===\n")

for p in recipe_prompts:
    ft = generate_text(model2, tokenizer2, p)
    print(f"Prompt: {p}")
    print(f"Baseline:   {baseline2[p][:120]}")
    print(f"Fine-Tuned: {ft[:120]}\n")

Perplexity: 73.88

=== FINE-TUNED RECIPES ===

Prompt: To make butter chicken
Baseline:   To make butter chicken.

This post may contain links to Amazon or other partners; your purchases via these links can ben
Fine-Tuned: To make butter chicken breast pieces into thin pieces. Cook until golden brown. Add chicken sauce and cook until golden 

Prompt: For pasta carbonara
Baseline:   For pasta carbonara pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta past
Fine-Tuned: For pasta carbonara stir fry onions and cook until golden. Add water and cook until vegetables start to combine. Add che

Prompt: To prepare a chocolate cake
Baseline:   To prepare a chocolate cake. For a cookie, go to the baking sheet, place a small cookie sheet on top, and place the cook
Fine-Tuned: To prepare a chocolate cake with eggs and stir well until golden. Preheat oven to 350 degrees. Chop egg mixture together



## Save Product Review Results

In [12]:
import pandas as pd

results_reviews = []

for p in review_prompts:
    ft = generate_text(model, tokenizer, p)

    results_reviews.append({
        "Prompt": p,
        "Baseline": baseline[p],
        "Fine_Tuned": ft
    })

df_reviews = pd.DataFrame(results_reviews)

# Save as CSV
df_reviews.to_csv("product_review_results.csv", index=False)

print("Saved product review results!")
df_reviews

Saved product review results!


,Prompt,Baseline,Fine_Tuned
0,This product is,This product is not designed for a specific pu...,This product is rated 4.4 out of 5 by 3 .\nRat...
1,I bought this phone and,I bought this phone and I can't wait to get it...,I bought this phone and I'm very excited about...
2,The quality of this item,The quality of this item will depend on your c...,The quality of this item has been outstanding ...


## Save Recipe Generation Results

In [13]:
results_recipes = []

for p in recipe_prompts:
    ft = generate_text(model2, tokenizer2, p)

    results_recipes.append({
        "Prompt": p,
        "Baseline": baseline2[p],
        "Fine_Tuned": ft
    })

df_recipes = pd.DataFrame(results_recipes)

df_recipes.to_csv("recipe_results.csv", index=False)

print("Saved recipe results!")
df_recipes

Saved recipe results!


,Prompt,Baseline,Fine_Tuned
0,To make butter chicken,To make butter chicken.\n\nThis post may conta...,To make butter chicken breast add garlic paste...
1,For pasta carbonara,For pasta carbonara pasta pasta pasta pasta pa...,For pasta carbonara boil pasta until well inco...
2,To prepare a chocolate cake,"To prepare a chocolate cake. For a cookie, go ...",To prepare a chocolate cake with butter and co...
